# Patent phrase matching: final offline run

Utkrisht Suman · A24ARIU0001

Executed on Kaggle T4, 9 November 2026, with internet off. All 11 code cells
completed without errors. Final version used for the competition submission:
https://www.kaggle.com/code/utkrishtsuman/a24ariu0001-patent-final?scriptVersionId=367835971

The configuration is loaded from the completed experiment output and checked
against the fixed budget. Section-name context was selected on grouped validation
before submission. The saved visible-test run obtained validation Pearson
0.820194478 and wrote 36 rows, with IDs in official sample-submission order.
The code uses the actual test length when Kaggle substitutes its hidden test set.

Adapted from the course `patent_phrase_starter.ipynb` and supplied continuation
package. Attached inputs are official competition data, offline weights/wheels
and the completed experiment output. No Hub fallback or hidden test labels are
used. Code cells and outputs below are unchanged from the saved run.


In [1]:
"""Prepare the offline patent runtime without replacing compatible image packages."""
from pathlib import Path
import importlib
import json
import subprocess
import sys
from importlib import metadata

ROOTS = [
    "transformers==5.22.2", "accelerate==1.15.0", "datasets==5.1.0",
    "scikit-learn>=1.4", "sentencepiece==0.2.3", "protobuf",
    "pandas>=2.1", "matplotlib>=3.8",
]

if Path("/kaggle/input").exists():
    candidates = sorted(Path("/kaggle/input").rglob("patent_runtime_wheels/wheels_manifest.json"))
    assert candidates, "Attach PATENT_DOWNLOAD_WEIGHTS output, including patent_runtime_wheels."
    manifest_path = candidates[0]
    manifest = json.loads(manifest_path.read_text())
    assert manifest["python"].split(".")[:2] == [str(sys.version_info.major), str(sys.version_info.minor)], \
        "The weights/wheels and notebook need the same Python major/minor version."
    roots = manifest.get("root_requirements", ROOTS)
    torch_before = metadata.version("torch")
    constraints = Path("/kaggle/working/patent_keep_cuda_torch.txt")
    constraints.write_text("torch==" + torch_before + "\n")
    base = [sys.executable, "-m", "pip", "install", "--disable-pip-version-check",
            "--no-index", "--find-links", str(manifest_path.parent), "-c", str(constraints)]
    report_path = Path("/kaggle/working/patent_runtime_plan.json")
    subprocess.run([*base, "--dry-run", "--report", str(report_path), *roots], check=True)
    planned = json.loads(report_path.read_text()).get("install", [])
    print("Offline package changes:", [p["metadata"]["name"] for p in planned], flush=True)

    # A loaded, unchanged dependency does not require a restart. Check the actual
    # module files belonging to distributions that pip proposes to replace.
    loaded_changes = {}
    loaded_files = {}
    for name, module in list(sys.modules.items()):
        origin = getattr(module, "__dict__", {}).get("__file__") if module else None
        if origin:
            loaded_files[name] = Path(origin).resolve()
    for package in planned:
        name = package["metadata"]["name"]
        try:
            dist = metadata.distribution(name)
        except metadata.PackageNotFoundError:
            continue
        files = {Path(dist.locate_file(f)).resolve() for f in dist.files or []}
        loaded = sorted(name for name, path in loaded_files.items() if path in files)
        if loaded:
            loaded_changes[name] = loaded[:8]
    if loaded_changes:
        raise RuntimeError("A fresh kernel is required BEFORE replacing loaded packages: "
                           + json.dumps(loaded_changes, sort_keys=True))
    if planned:
        subprocess.run([*base, *roots], check=True)
        importlib.invalidate_caches()
    assert metadata.version("torch") == torch_before, "Kaggle CUDA torch must be preserved."
    print("Offline runtime ready; compatible packages and CUDA torch preserved.", flush=True)
else:
    print("Local patent run: install requirements.txt and set PATENT_* paths first.")


Looking in links: /kaggle/input/notebooks/utkrishtsuman/a24ariu0001-patent-offline-weights/patent_runtime_wheels
Processing /kaggle/input/notebooks/utkrishtsuman/a24ariu0001-patent-offline-weights/patent_runtime_wheels/transformers-5.22.2-py3-none-any.whl
Processing /kaggle/input/notebooks/utkrishtsuman/a24ariu0001-patent-offline-weights/patent_runtime_wheels/datasets-5.1.0-py3-none-any.whl
Processing /kaggle/input/notebooks/utkrishtsuman/a24ariu0001-patent-offline-weights/patent_runtime_wheels/sentencepiece-0.2.3-cp313-cp313-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl
Processing /kaggle/input/notebooks/utkrishtsuman/a24ariu0001-patent-offline-weights/patent_runtime_wheels/pyarrow-26.0.1-cp313-cp313-manylinux_2_28_x86_64.whl (from datasets==5.1.0)
Would install datasets-5.1.0 pyarrow-26.0.1 sentencepiece-0.2.3 transformers-5.22.2
Offline package changes: ['transformers', 'datasets', 'sentencepiece', 'pyarrow']
Looking in links: /kaggle/input/notebooks/utkrishtsuman/a24ariu0001-pate

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.51.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarrow 26.0.1 which is incompatible.


Offline runtime ready; compatible packages and CUDA torch preserved.


## 1. Setup

In [2]:
import glob
import os
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")

import numpy as np
import pandas as pd
import torch
import transformers
from datasets import Dataset
from sklearn.model_selection import GroupShuffleSplit
from tokenizers.processors import TemplateProcessing
from transformers import (AutoModelForSequenceClassification, AutoTokenizer,
                          DataCollatorWithPadding, Trainer, TrainingArguments)

SEED = 42
transformers.set_seed(SEED)   # seeds Python, NumPy, and PyTorch; GPU results can still vary slightly between runs
device = "cuda" if torch.cuda.is_available() else "cpu"
print("torch", torch.__version__, "| transformers", transformers.__version__, "| device:", device)
import inspect
# Preserve the starter 10% warmup across Transformers API versions.
WARMUP = ({"warmup_ratio": 0.1} if "warmup_ratio" in inspect.signature(TrainingArguments).parameters
          else {"warmup_steps": 0.1})

if Path("/kaggle/working").exists():
    assert torch.cuda.is_available(), "Enable a Kaggle T4 GPU before running the patent training notebook."


torch 2.11.0+cu130 | transformers 5.22.2 | device: cuda


## 2. Validation-selected configuration

`best_config.json` selects the section baseline by validation Pearson. Seed 42,
three epochs, learning rate 2e-5, batch 32, length 64 and a 20% anchor-group
holdout are checked before training. Leaderboard scores do not select settings.


In [3]:
import json, time, gc
from pathlib import Path
MODEL_ID = "microsoft/deberta-v3-small"
MAX_LEN, EPOCHS, LR, BATCH_SIZE, VAL_FRACTION = 64, 3, 2e-5, 32, 0.2
OUT = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("patent_outputs")
OUT.mkdir(parents=True, exist_ok=True)
OUT_DIR = str(OUT)
def find_one(pattern):
    hits = sorted(glob.glob(f"/kaggle/input/**/{pattern}", recursive=True))
    return os.path.dirname(hits[0]) if hits else None
DATA_DIR = os.environ.get("PATENT_DATA_DIR") or find_one("us-patent-phrase-to-phrase-matching/train.csv")
assert DATA_DIR, "Attach the competition data or set PATENT_DATA_DIR to its local folder."
MODEL_PATH = os.environ.get("PATENT_MODEL_PATH") or find_one("deberta-v3-small/config.json")
assert MODEL_PATH, "Attach offline pretrained weights or set PATENT_MODEL_PATH; no Hub fallback."
print("data:", DATA_DIR, "| offline model:", MODEL_PATH)

data: /kaggle/input/competitions/us-patent-phrase-to-phrase-matching | offline model: /kaggle/input/notebooks/utkrishtsuman/a24ariu0001-patent-offline-weights/deberta-v3-small


## 3. Data

Each row asks how similar a **target** phrase is to an **anchor** phrase, judged within a patent **context**. The context is a CPC (Cooperative Patent Classification) code such as `G02`, whose first letter is the section. Scores are 0, 0.25, 0.5, 0.75, or 1. The visible `test.csv` has only a few rows; when you submit, Kaggle reruns the notebook with the hidden test set in its place.

In [4]:
train = pd.read_csv(f"{DATA_DIR}/train.csv")
test = pd.read_csv(f"{DATA_DIR}/test.csv")
print("train:", train.shape, "| test (visible):", test.shape)
print(train.score.value_counts().sort_index())
train.head()

train: (36473, 5) | test (visible): (36, 4)
score
0.00     7471
0.25    11519
0.50    12300
0.75     4029
1.00     1154
Name: count, dtype: int64


,id,anchor,target,context,score
0,37d61fd2272659b1,abatement,abatement of pollution,A47,0.50
1,7b9652b17b68b7a4,abatement,act of abating,A47,0.75
2,36d72442aefd8232,abatement,active catalyst,A47,0.25
3,5296b0c19e1ce60e,abatement,eliminating process,A47,0.50
4,54c1e3b9184cb5b6,abatement,forest region,A47,0.00


## 4. Model input

The tokenizer encodes the anchor as one segment and the target plus CPC section
name as the other. Joint attention makes this a cross-encoder. The selected
context is the broad section description, not a full CPC class title.


In [5]:
config_path = os.environ.get("PATENT_CONFIG_PATH") or next(iter(sorted(
    glob.glob("/kaggle/input/**/best_config.json", recursive=True))), None)
assert config_path, "Attach the experiment output with best_config.json; do not choose by leaderboard score."
selection = json.loads(Path(config_path).read_text())
expected = {"seed": SEED, "epochs": EPOCHS, "lr": LR, "batch_size": BATCH_SIZE,
            "max_len": MAX_LEN, "val_fraction": VAL_FRACTION, "model_id": MODEL_ID}
assert all(selection.get(k) == v for k, v in expected.items()), "Final settings must match experiments"
assert selection.get("selected_by") == "validation Pearson"
CONTEXT_MODE = selection["context_mode"]
CPC_SECTIONS = {"A": "human necessities", "B": "performing operations; transporting",
    "C": "chemistry; metallurgy", "D": "textiles; paper", "E": "fixed constructions",
    "F": "mechanical engineering; lighting; heating; weapons; blasting", "G": "physics", "H": "electricity"}
def add_inputs(df):
    df = df.copy(); df["text_a"] = df.anchor
    if CONTEXT_MODE == "section": df["text_b"] = df.target + "; " + df.context.str[0].map(CPC_SECTIONS)
    elif CONTEXT_MODE == "none": df["text_b"] = df.target
    elif CONTEXT_MODE == "code": df["text_b"] = df.target + "; " + df.context
    else: raise ValueError(CONTEXT_MODE)
    return df
train, test = add_inputs(train), add_inputs(test)
print("Chosen from validation:", selection)

Chosen from validation: {'context_mode': 'section', 'selected_by': 'validation Pearson', 'name': 'section_baseline', 'validation_pearson': 0.8201944782513845, 'seed': 42, 'epochs': 3, 'lr': 2e-05, 'batch_size': 32, 'max_len': 64, 'val_fraction': 0.2, 'model_id': 'microsoft/deberta-v3-small'}


## 5. Validation split

The fixed grouped split has 28,914 training and 7,559 validation rows, with no
anchor overlap. Keeping the same split permits comparison with the experiment
run. It also keeps the same training budget rather than silently retraining on
all labels after selecting settings.


In [6]:
splitter = GroupShuffleSplit(n_splits=1, test_size=VAL_FRACTION, random_state=SEED)
train_idx, val_idx = next(splitter.split(train, groups=train["anchor"]))
trn, val = train.iloc[train_idx].reset_index(drop=True), train.iloc[val_idx].reset_index(drop=True)
assert set(trn.anchor).isdisjoint(val.anchor)
print(f"train rows {len(trn)} | val rows {len(val)} | val anchors {val.anchor.nunique()}")

train rows 28914 | val rows 7559 | val anchors 147


## 6. Tokenization

Padding is left to the data collator, which pads each batch only to its longest sequence. The printed example should start with `[CLS]` and have a `[SEP]` between the two segments.

In [7]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, local_files_only=True)
# transformers 5.0 builds this tokenizer without [CLS]/[SEP]; set the pair template explicitly
tokenizer.backend_tokenizer.post_processor = TemplateProcessing(
    single="[CLS] $A [SEP]", pair="[CLS] $A [SEP] $B:1 [SEP]:1",
    special_tokens=[("[CLS]", tokenizer.cls_token_id), ("[SEP]", tokenizer.sep_token_id)])


def to_dataset(df, with_labels=True):
    cols = ["text_a", "text_b"] + (["score"] if with_labels else [])
    ds = Dataset.from_pandas(df[cols].astype({"score": "float32"}) if with_labels else df[cols])  # MSE loss needs float32 labels
    ds = ds.map(lambda b: tokenizer(b["text_a"], b["text_b"], truncation=True, max_length=MAX_LEN),
                batched=True, remove_columns=["text_a", "text_b"])
    return ds.rename_column("score", "labels") if with_labels else ds


ds_trn, ds_val, ds_test = to_dataset(trn), to_dataset(val), to_dataset(test, with_labels=False)
print(tokenizer.decode(ds_trn[0]["input_ids"]))

[transformers] The tokenizer you are loading from '/kaggle/input/notebooks/utkrishtsuman/a24ariu0001-patent-offline-weights/deberta-v3-small' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.


Map:   0%|          | 0/28914 [00:00<?, ? examples/s]

Map:   0%|          | 0/7559 [00:00<?, ? examples/s]

Map:   0%|          | 0/36 [00:00<?, ? examples/s]

[CLS] abatement[SEP] abatement of pollution; human necessities[SEP]


## 7. Fine-tuning

With `num_labels=1`, the classification head outputs a single number. Because the labels are floats, Hugging Face treats the task as **regression** and minimizes mean squared error. The competition metric is the **Pearson correlation** between predictions and true scores. Pearson correlation ignores any linear rescaling of the predictions, so the model is rewarded for ranking pairs in the right order, not for hitting the exact values.

In [8]:
def compute_metrics(eval_pred):
    preds, labels = eval_pred
    return {"pearson": float(np.corrcoef(preds.reshape(-1), labels)[0, 1])}


# Load in fp32: the deberta-v3 checkpoints are stored in fp16, and fp16=True mixed precision below needs
# fp32 weights ("Attempting to unscale FP16 gradients" otherwise). Older transformers call this torch_dtype.
model = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH, num_labels=1, dtype=torch.float32, local_files_only=True)
args = TrainingArguments(
    output_dir=f"{OUT_DIR}/trainer_tmp",
    num_train_epochs=EPOCHS,
    learning_rate=LR,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=2 * BATCH_SIZE,
    **WARMUP,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="no",          # keeps disk use and runtime low; the final weights stay in memory
    logging_steps=100,
    fp16=torch.cuda.is_available(),
    report_to="none",
    seed=SEED,
)
trainer = Trainer(model=model, args=args, train_dataset=ds_trn, eval_dataset=ds_val,
                  data_collator=DataCollatorWithPadding(tokenizer), compute_metrics=compute_metrics)
trainer.train()

Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

[transformers] DebertaV2ForSequenceClassification LOAD REPORT from: /kaggle/input/notebooks/utkrishtsuman/a24ariu0001-patent-offline-weights/deberta-v3-small
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.classifier.bias        | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
classifier.bias                         | MISSING    | 
pooler.dense.bias                       | MISSING    | 
cl

Epoch,Training Loss,Validation Loss,Pearson
1,0.029498,0.027667,0.803232
2,0.022482,0.024409,0.819270
3,0.019423,0.024366,0.820194


TrainOutput(global_step=2712, training_loss=0.03035423632128752, metrics={'train_runtime': 214.4694, 'train_samples_per_second': 404.449, 'train_steps_per_second': 12.645, 'total_flos': 440284446202860.0, 'train_loss': 0.03035423632128752, 'epoch': 3.0})

## 8. Validation predictions

The predictions are saved so you can do your error analysis without retraining.

In [9]:
val["pred"] = trainer.predict(ds_val).predictions.reshape(-1)
val_pearson = np.corrcoef(val["pred"], val["score"])[0, 1]
print(f"Validation Pearson: {val_pearson:.4f}")
val[["id", "anchor", "target", "context", "score", "pred"]].to_csv(f"{OUT_DIR}/val_predictions.csv", index=False)

Validation Pearson: 0.8202


## 9. Submission

The output is `/kaggle/working/submission.csv`, with exactly `id,score` and one
row for every test record. Scores are clipped to [0, 1]. Validation predictions
remain raw for the error and calibration analysis; clipping can change Pearson.


In [10]:
test_pred = trainer.predict(ds_test).predictions.reshape(-1)
submission = pd.DataFrame({"id": test["id"], "score": np.clip(test_pred, 0, 1)})
assert len(submission) == len(test) and submission["score"].notna().all()
submission.to_csv(f"{OUT_DIR}/submission.csv", index=False)
submission.head()

,id,score
0,4112d61851461f60,0.566895
1,09e418c93a776564,0.786133
2,36baf228038e314b,0.498779
3,1f37ead645e7f0c8,0.327881
4,71a5b6ad068d531f,0.000000


## Results and submission

The saved run reached validation Pearson **0.820194478** on 7,559 rows. `Trainer` recorded 214.4694 seconds (3.5745 minutes) for training and epoch evaluations. The full model-loading and training cell took 272.2310 seconds, so those timings cover different work. Training finished at step 2,712 after three epochs.

Kaggle submission **58176236** completed from this notebook version on 9 November 2026. Its public score is **0.7922** and private score is **0.8084**. These are below validation by 0.0280 and 0.0118 respectively. Different anchor and section mixes, partition sizes and selection on validation can contribute to the gap. I kept the configuration selected on validation and made one competition submission; the experiment baseline CSV was not submitted.

The official API response and ledger are saved in `A24ARIU0001_results/`. A genuine Submissions-tab screenshot is still required. The four-page report remains a draft until that image is added, and course-platform delivery is pending.

References: course starter and project brief; He et al. (2021), arXiv:2111.09543; Kaggle U.S. Patent Phrase to Phrase Matching; `microsoft/deberta-v3-small` model card (MIT).


## Download the result ZIP
The next cell collects small outputs only. Also download this notebook **with outputs** after the saved run finishes; Kaggle does not place its notebook file in /kaggle/working automatically.

In [11]:
#!/usr/bin/env python3
"""Collect review outputs without datasets, environments or model weights. Standard library only."""
import argparse
from pathlib import Path
import zipfile
import json
import hashlib

SKIP_DIRS={'.venv','venv','.git','__pycache__','.cache','node_modules','site-packages',
           'patent_runtime_wheels','deberta-v3-small','all-MiniLM-L6-v2','data','adapter',
           'trainer','trainer_tmp','patent_weights','reference_evidence'}
JSON_NAMES={'best_config.json','run_summary.json','test_metrics.json','history.json','CHECKS.json'}
SKIP_FILES={'train.csv','test.csv','sample_submission.csv','IndianFoodDatasetCSV.csv',
            'tokenizer.json','config.json','tokenizer_config.json','adapter_config.json'}
SUFFIXES={'.ipynb','.csv','.png','.jpg','.jpeg','.pdf','.log','.txt'}

def eligible(path):
    if path.is_symlink() or not path.is_file():return False
    if any(p in SKIP_DIRS or p.startswith('checkpoint-') for p in path.parts):return False
    if path.name in SKIP_FILES:return False
    if path.suffix.lower()=='.json':return path.name in JSON_NAMES
    if path.suffix.lower() not in SUFFIXES:return False
    if path.suffix.lower()=='.txt' and path.name not in {'notebook_urls.txt','errors.txt','review_notes.txt'}:return False
    return True

def collect(roots,destination):
    destination=destination.resolve();destination.parent.mkdir(parents=True,exist_ok=True)
    count=0;manifest=[]
    with zipfile.ZipFile(destination,'w',zipfile.ZIP_DEFLATED) as archive:
        for index,root in enumerate(roots,1):
            root=root.resolve()
            if not root.exists():raise FileNotFoundError(root)
            paths=[root] if root.is_file() else sorted(root.rglob('*'))
            for path in paths:
                if path.resolve()==destination or not eligible(path):continue
                relative=Path(path.name) if root.is_file() else path.relative_to(root)
                arc=Path(f'{index:02d}_{root.stem}')/relative
                archive.write(path,str(arc));count+=1
                manifest.append({'path':str(arc),'bytes':path.stat().st_size,
                                 'sha256':hashlib.sha256(path.read_bytes()).hexdigest()})
        archive.writestr('RETURN_MANIFEST.json',json.dumps(manifest,indent=2))
    if count==0:
        destination.unlink();raise ValueError('No review outputs found. Check the input folder.')
    print(f'Saved {count} files: {destination} ({destination.stat().st_size/1e6:.2f} MB)',flush=True)


if Path("/kaggle/working").exists():
    collect([Path("/kaggle/working")],Path("/kaggle/working/A24ARIU0001_PATENT_FINAL_RESULTS.zip"))


Saved 3 files: /kaggle/working/A24ARIU0001_PATENT_FINAL_RESULTS.zip (0.20 MB)
